# Hypothesis Testing for EDA: Exercise

**Notebook 5 of 5.** The last one, and the only one without a worked example to lean on.

You are exploring the `taxis` dataset, saved in `data/taxis.csv`: 6,433 New York City taxi rides with the distance, fare, tip, payment method, cab colour, and pickup/dropoff location of each ride. As in the EDA guide, every pattern you spot needs a test before it goes in your report.

For each exercise below, follow the same recipe as the guide notebook:

1. **Plot** the pair of variables.
2. **Quantify** the pattern with an effect size (correlation coefficient, difference in means, Cramér's V).
3. **Test** it with the matching hypothesis test.
4. **Interpret** the result in plain language: is the pattern real, and does it matter?

> **Note.** This dataset is realistically messy: `payment` and the borough columns contain some missing values. Decide how to handle them before testing (dropping the affected rows is fine, state that you did it).

## Learning Objectives

At the end of this notebook, you should be able to:

- Select and run the matching association test for each variable-pair type without a template.
- Check test assumptions on messy real data and switch to a robust alternative when they fail.
- Distinguish statistical significance from practical relevance using effect sizes.
- Spot patterns that a test confirms but a data-recording artifact explains.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

df = pd.read_csv("data/taxis.csv", parse_dates=["pickup", "dropoff"])
df.head()

---
## Exercise 1: Do Longer Rides Cost More?

Numeric vs numeric: `distance` and `fare`.

- Make the scatter plot.
- Compute the Pearson correlation and its p-value. Compare with Spearman.
- Interpret the coefficient and the p-value together. Is this finding interesting for a stakeholder, or just a sanity check of how taxi meters work?

In [ ]:
# 1. Plot: scatter plot of distance vs fare
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=df, x="distance", y="fare", alpha=0.3, ax=ax)
ax.set_title("Distance vs fare")
ax.set_xlabel("Distance (miles)")
ax.set_ylabel("Fare (USD)")
plt.show()

# 2. Quantify + 3. Test: Pearson (linear) and Spearman (rank-based, robust to outliers)
pearson_r, pearson_p = stats.pearsonr(df["distance"], df["fare"])
spearman_rho, spearman_p = stats.spearmanr(df["distance"], df["fare"])

print(f"Pearson r    = {pearson_r:.3f}, p = {pearson_p:.3g}")
print(f"Spearman rho = {spearman_rho:.3f}, p = {spearman_p:.3g}")

# Rides with zero distance still have a fare (base fee, or a recording artifact?)
print(f"Rides with distance == 0: {(df['distance'] == 0).sum()}")

Your interpretation here:

---
## Exercise 2: Do Card Payers Tip More Than Cash Payers?

Categorical (2 groups) vs numeric: `payment` and `tip`.

- Plot the tip distribution per payment method and compute the group means.
- Run the appropriate two-group test.
- The test will scream "significant". Before you write the conclusion, look at the cash tips themselves. What is really going on, and can any hypothesis test detect it?

> **Tip.** A p-value tells you the pattern is not random noise. It cannot tell you the pattern means what you think it means. Look at *how* this dataset records cash tips.

In [ ]:
# Drop the 44 rides with missing payment method
df_pay = df.dropna(subset=["payment"])

# 1. Plot: tip distribution per payment method
fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=df_pay, x="payment", y="tip", ax=ax)
ax.set_title("Tip by payment method")
ax.set_ylabel("Tip (USD)")
plt.show()

# 2. Quantify: group means (and medians, tips are skewed)
print(df_pay.groupby("payment")["tip"].agg(["mean", "median", "count"]))

# 3. Test: Welch's t-test (no equal-variance assumption) + Mann-Whitney U as robust check
card_tips = df_pay.loc[df_pay["payment"] == "credit card", "tip"]
cash_tips = df_pay.loc[df_pay["payment"] == "cash", "tip"]

t_stat, t_p = stats.ttest_ind(card_tips, cash_tips, equal_var=False)
u_stat, u_p = stats.mannwhitneyu(card_tips, cash_tips)
print(f"Welch t = {t_stat:.2f}, p = {t_p:.3g}")
print(f"Mann-Whitney U = {u_stat:.0f}, p = {u_p:.3g}")

# Look at the cash tips themselves
print(cash_tips.describe())
print(f"Cash rides with a tip > 0: {(cash_tips > 0).sum()} of {len(cash_tips)}")

Your interpretation here:

### Exercise 2b (Trickier): Do Green Cabs Charge Higher Fares?

Same pair type: `color` and `fare`.

- Compare the mean **and** the median fare of yellow and green cabs.
- Run both Welch's t-test and the Mann-Whitney U test.
- The two tests disagree at alpha = 0.05, and the means and medians tell different stories. Explain what is happening and what you would report.

> **Tip.** When means differ but medians are identical, ask which part of the distribution is doing the work, and remember the two tests are not asking exactly the same question.

In [ ]:
yellow_fare = df.loc[df["color"] == "yellow", "fare"]
green_fare = df.loc[df["color"] == "green", "fare"]

# 1. Plot: fare distribution per cab colour (log scale to see the right tail)
fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=df, x="color", y="fare", ax=ax)
ax.set_yscale("log")
ax.set_title("Fare by cab colour")
ax.set_ylabel("Fare (USD, log scale)")
plt.show()

# 2. Quantify: mean vs median, plus quartiles to see where the difference sits
print(df.groupby("color")["fare"].describe())
print(f"Difference in means: {green_fare.mean() - yellow_fare.mean():.2f} USD")

# 3. Test: Welch's t-test (compares means) vs Mann-Whitney U (compares ranks)
t_stat, t_p = stats.ttest_ind(green_fare, yellow_fare, equal_var=False)
u_stat, u_p = stats.mannwhitneyu(green_fare, yellow_fare)
print(f"Welch t = {t_stat:.2f}, p = {t_p:.4f}")
print(f"Mann-Whitney U = {u_stat:.0f}, p = {u_p:.4f}")

Your interpretation here:

---
## Exercise 3: Does the Fare Depend on the Pickup Borough?

Categorical (4 groups) vs numeric: `pickup_borough` and `fare`.

- Plot fare per borough and compute the group means.
- Run the test that compares all four groups at once.
- Interpret the result. What would be the correct follow-up to find out **which** boroughs differ, and what must that follow-up guard against?

In [ ]:
# Drop the 26 rides with missing pickup borough
df_boro = df.dropna(subset=["pickup_borough"])

# 1. Plot: fare per pickup borough
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df_boro, x="pickup_borough", y="fare", ax=ax)
ax.set_title("Fare by pickup borough")
ax.set_ylabel("Fare (USD)")
plt.show()

# 2. Quantify: group means, medians, spread
print(df_boro.groupby("pickup_borough")["fare"].agg(["mean", "median", "std", "count"]))

# Assumption checks for ANOVA: equal variances and roughly normal groups
groups = [g["fare"].values for _, g in df_boro.groupby("pickup_borough")]
lev_stat, lev_p = stats.levene(*groups)
print(f"Levene (equal variances): p = {lev_p:.3g}")
print("Skewness per borough:", df_boro.groupby("pickup_borough")["fare"].skew().round(2).to_dict())

# 3. Test: one-way ANOVA, plus Kruskal-Wallis because the assumptions fail
f_stat, f_p = stats.f_oneway(*groups)
h_stat, h_p = stats.kruskal(*groups)
print(f"ANOVA F = {f_stat:.1f}, p = {f_p:.3g}")
print(f"Kruskal-Wallis H = {h_stat:.1f}, p = {h_p:.3g}")

# Follow-up: pairwise Mann-Whitney tests with Bonferroni correction (6 comparisons)
from itertools import combinations

boroughs = sorted(df_boro["pickup_borough"].unique())
pairs = list(combinations(boroughs, 2))
for a, b in pairs:
    _, p = stats.mannwhitneyu(
        df_boro.loc[df_boro["pickup_borough"] == a, "fare"],
        df_boro.loc[df_boro["pickup_borough"] == b, "fare"],
    )
    p_adj = min(p * len(pairs), 1.0)
    print(f"{a:>9} vs {b:<9}: p_adj = {p_adj:.3g}")

Your interpretation here:

---
## Exercise 4: Do Yellow and Green Cabs Work Different Boroughs?

Categorical vs categorical: `color` and `pickup_borough`.

- Make the count plot and build the contingency table.
- Run the chi-square test of independence and compute Cramér's V.
- Interpret both numbers. Then repeat for `payment` vs `color` and compare the two effect sizes: which association is strong and which is significant-but-weak?

> **Tip.** New York licenses green cabs specifically for the boroughs outside the Manhattan core. Does the data show it?

In [ ]:
def cramers_v(table):
    """Chi-square test of independence plus Cramér's V for a contingency table."""
    chi2, p, dof, expected = stats.chi2_contingency(table)
    n = table.to_numpy().sum()
    v = np.sqrt(chi2 / (n * (min(table.shape) - 1)))
    print(f"chi2 = {chi2:.1f}, dof = {dof}, p = {p:.3g}, Cramér's V = {v:.3f}")
    print(f"Expected counts < 5: {(expected < 5).sum()}")


# --- color vs pickup_borough (drop rides with missing borough) ---
df_boro = df.dropna(subset=["pickup_borough"])

# 1. Plot: count plot
fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(data=df_boro, x="pickup_borough", hue="color", palette={"yellow": "gold", "green": "seagreen"}, ax=ax)
ax.set_title("Pickups per borough by cab colour")
plt.show()

# 2. Contingency table (counts and row shares)
boro_table = pd.crosstab(df_boro["color"], df_boro["pickup_borough"])
print(boro_table)
print(pd.crosstab(df_boro["color"], df_boro["pickup_borough"], normalize="index").round(2))

# 3. Test + effect size
cramers_v(boro_table)

# --- payment vs color (drop rides with missing payment) ---
df_pay = df.dropna(subset=["payment"])
pay_table = pd.crosstab(df_pay["color"], df_pay["payment"])
print(pay_table)
print(pd.crosstab(df_pay["color"], df_pay["payment"], normalize="index").round(2))
cramers_v(pay_table)

Your interpretation here:

---
## Exercise 5: Reflection, the Multiple-Testing Ledger

No code needed. Count the tests you ran in this notebook (including assumption checks), then answer:

1. At alpha = 0.05, roughly how worried should you be about false positives across that many tests?
2. Which of your findings would you report as solid, which as leads needing confirmation, and which as artifacts of how the data was recorded? Justify each with the p-value **and** the effect size.

Your answers here:

---
## Summary

In this notebook you:

- Selected and ran the matching test for four variable-pair questions on messy real data.
- Checked assumptions and switched to robust alternatives where normality failed.
- Separated statistical significance from practical relevance using effect sizes.
- Caught a pattern that the test confirms but a data-recording artifact explains.

Compare your approach with the worked reference, `solutions/05_hypothesis_testing_for_eda_exercise.ipynb`, once you have finished, not before.

## References & Further Reading

- [**scipy.stats reference**](https://docs.scipy.org/doc/scipy/reference/stats.html): Every test you need for these exercises.
- [**scipy.stats.chi2_contingency**](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chi2_contingency.html): Chi-square test of independence.
- [**Codecademy: Testing an Association cheat sheet**](https://www.codecademy.com/learn/hypothesis-testing-python/modules/hp-testing-an-association/cheatsheet): One-page recap of which test fits which pair.
- [**StatQuest: P-hacking**](https://www.youtube.com/watch?v=HDCOUXE3HMM): Watch before writing your Exercise 5 answers.